# **Proyecto 3 – Simulación de Carrera F1 en CUDA**

## **Instalación extensión CUDA (nvcc4jupyter)**

In [9]:
!pip -q install nvcc4jupyter
%load_ext nvcc4jupyter

The nvcc4jupyter extension is already loaded. To reload it, use:
  %reload_ext nvcc4jupyter


## **Detección simplificada de GPU**

In [10]:
import subprocess, re
info = subprocess.getoutput('nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader,nounits')
if 'not found' in info.lower() or not info.strip():
    print('GPU no detectada: se mostrará fallback si falla CUDA.')
    arch='sm_70'
else:
    try:
        # Expected format: "GPU_Name, COMPUTE_CAPABILITY" e.g., "Tesla T4, 7.5"
        parts = info.split(',')
        if len(parts) > 1:
            compute_cap_str = parts[1].strip() # "7.5"
            arch = 'sm_' + compute_cap_str.replace('.', '') # "sm_75"
            print(f'GPU detectada: {parts[0].strip()} con arquitectura {arch}')
        else:
            print(f'GPU detectada, pero no se pudo determinar la arquitectura: {info}. Usando sm_70.')
            arch = 'sm_70'
    except Exception as e:
        print(f'Error al parsear info de GPU: {e}. Usando sm_70.')
        arch = 'sm_70'

GPU detectada: Tesla T4 con arquitectura sm_75


## **Kernels CUDA y funciones (Global, Const, Shared)**

In [11]:
%%cuda -arch={arch}
#include <cstdio>
// Simplified CUDA F1 Simulation Kernels
// Variants: GlobalOnly, ConstGlobal, ConstSharedGlobal
// Outputs: RESULT lines for benchmarking + RACE_SUMMARY
// ----------------------------------------------------
// Parameters
static const int N = 16; // fewer pilots to keep simple
static const int LAPS = 20;
static const int SECT = 4;
static const int REPS = 10;
struct Stat { int best; int total; };
// ---------------- Global-only kernel ----------------
__global__ void kGlobal(Stat *stats, int *bestGlobal){
  int stride = blockDim.x * gridDim.x;
  for(int idx = blockIdx.x*blockDim.x + threadIdx.x; idx < N*LAPS; idx += stride){
    int p = idx / LAPS; int lap = idx % LAPS; unsigned int seed = 1234u + idx*13u;
    int lapTime=0;
    for(int s=0; s<SECT; ++s){ seed=seed*1103515245u+12345u; lapTime += 25 + ((seed>>16)%7); }
    seed=seed*1103515245u+12345u; float r=((seed>>8)&0xFFFF)/65535.0f; if(r<0.05f) lapTime += 15;
    atomicAdd(&stats[p].total, lapTime); atomicMin(&stats[p].best, lapTime); atomicMin(bestGlobal, lapTime);
  }
}
// ---------------- Constant-memory kernel -------------
__constant__ int cBase[8]; __constant__ int cPit[3]; __constant__ float cProb;
__global__ void kConst(Stat *stats, int *bestGlobal){
  int stride=blockDim.x*gridDim.x;
  for(int idx=blockIdx.x*blockDim.x+threadIdx.x; idx < N*LAPS; idx+=stride){
    int p=idx/LAPS; int lap=idx%LAPS; unsigned int seed=2222u + idx*17u; int lapTime=0;
    for(int s=0; s<SECT; ++s){ seed=seed*1103515245u+12345u; int v=(seed>>16)%7; lapTime += cBase[s] + v; }
    int stage=(lap < LAPS/3)?0:(lap < 2*LAPS/3?1:2); seed=seed*1103515245u+12345u; float r=((seed>>8)&0xFFFF)/65535.0f; if(r<cProb) lapTime += cPit[stage];
    atomicAdd(&stats[p].total, lapTime); atomicMin(&stats[p].best, lapTime); atomicMin(bestGlobal, lapTime);
  }
}
// ---------------- Shared-memory kernel ---------------
__constant__ int cBaseSh[8]; __constant__ int cPitSh[3]; __constant__ float cProbSh;
__global__ void kShared(Stat *stats, int *bestGlobal){
  extern __shared__ int sh[]; int *base = sh; int *partial = sh + SECT;
  if(threadIdx.x < SECT) base[threadIdx.x] = cBaseSh[threadIdx.x];
  __syncthreads();
  int stride=blockDim.x*gridDim.x; int localBest = 1000000000;
  for(int idx=blockIdx.x*blockDim.x+threadIdx.x; idx < N*LAPS; idx+=stride){
    int p=idx/LAPS; int lap=idx%LAPS; unsigned int seed=3333u + idx*19u + threadIdx.x; int lapTime=0;
    for(int s=0; s<SECT; ++s){ seed=seed*1103515245u+12345u; int v=(seed>>16)%7; lapTime += base[s] + v; }
    int stage=(lap < LAPS/3)?0:(lap < 2*LAPS/3?1:2); seed=seed*1103515245u+12345u; float r=((seed>>8)&0xFFFF)/65535.0f; if(r<cProbSh) lapTime += cPitSh[stage];
    atomicAdd(&stats[p].total, lapTime); atomicMin(&stats[p].best, lapTime); if(lapTime < localBest) localBest=lapTime;
  }
  partial[threadIdx.x] = localBest; __syncthreads();
  for(int off=blockDim.x/2; off>0; off >>=1){ if(threadIdx.x < off){ partial[threadIdx.x] = min(partial[threadIdx.x], partial[threadIdx.x+off]); } __syncthreads(); }
  if(threadIdx.x==0) atomicMin(bestGlobal, partial[0]);
}
// ------------- Helpers --------------------------------
__host__ void initStats(Stat *h){ for(int i=0;i<N;++i){ h[i].best=1000000000; h[i].total=0; } }
__host__ void benchmarkGlobal(){
  Stat *d_stats; int *d_best; cudaMalloc(&d_stats, N*sizeof(Stat)); cudaMalloc(&d_best,sizeof(int));
  dim3 block(256); dim3 grid((N*LAPS + block.x -1)/block.x); if(grid.x>1024) grid.x=1024;
  for(int r=0;r<REPS;++r){ Stat h[N]; initStats(h); int init=1000000000; cudaMemcpy(d_stats,h,N*sizeof(Stat),cudaMemcpyHostToDevice); cudaMemcpy(d_best,&init,sizeof(int),cudaMemcpyHostToDevice);
    cudaEvent_t a,b; cudaEventCreate(&a); cudaEventCreate(&b); cudaEventRecord(a); kGlobal<<<grid,block>>>(d_stats,d_best); cudaEventRecord(b); cudaEventSynchronize(b); float ms; cudaEventElapsedTime(&ms,a,b); int g; cudaMemcpy(&g,d_best,sizeof(int),cudaMemcpyDeviceToHost); printf("RESULT,GlobalOnly,%d,%.3f,%d\n", r+1, ms, g); cudaEventDestroy(a); cudaEventDestroy(b); }
  cudaFree(d_stats); cudaFree(d_best);
}
__host__ void benchmarkConst(){
  Stat *d_stats; int *d_best; cudaMalloc(&d_stats,N*sizeof(Stat)); cudaMalloc(&d_best,sizeof(int)); int hBase[8]; for(int i=0;i<SECT;++i) hBase[i]=25+(i%2); int hPit[3]={12,15,10}; float prob=0.06f; cudaMemcpyToSymbol(cBase,hBase,8*sizeof(int)); cudaMemcpyToSymbol(cPit,hPit,3*sizeof(int)); cudaMemcpyToSymbol(cProb,&prob,sizeof(float));
  dim3 block(256); dim3 grid((N*LAPS + block.x -1)/block.x); if(grid.x>1024) grid.x=1024;
  for(int r=0;r<REPS;++r){ Stat h[N]; initStats(h); int init=1000000000; cudaMemcpy(d_stats,h,N*sizeof(Stat),cudaMemcpyHostToDevice); cudaMemcpy(d_best,&init,sizeof(int),cudaMemcpyHostToDevice); cudaEvent_t a,b; cudaEventCreate(&a); cudaEventCreate(&b); cudaEventRecord(a); kConst<<<grid,block>>>(d_stats,d_best); cudaEventRecord(b); cudaEventSynchronize(b); float ms; cudaEventElapsedTime(&ms,a,b); int g; cudaMemcpy(&g,d_best,sizeof(int),cudaMemcpyDeviceToHost); printf("RESULT,ConstGlobal,%d,%.3f,%d\n", r+1, ms, g); cudaEventDestroy(a); cudaEventDestroy(b);}
  cudaFree(d_stats); cudaFree(d_best);
}
__host__ void benchmarkShared(){
  Stat *d_stats; int *d_best; cudaMalloc(&d_stats,N*sizeof(Stat)); cudaMalloc(&d_best,sizeof(int)); int hBase[8]; for(int i=0;i<SECT;++i) hBase[i]=25+(i%3); int hPit[3]={10,14,11}; float prob=0.055f; cudaMemcpyToSymbol(cBaseSh,hBase,8*sizeof(int)); cudaMemcpyToSymbol(cPitSh,hPit,3*sizeof(int)); cudaMemcpyToSymbol(cProbSh,&prob,sizeof(float));
  dim3 block(256); dim3 grid((N*LAPS + block.x -1)/block.x); if(grid.x>2048) grid.x=2048; size_t shBytes=(SECT + block.x)*sizeof(int);
  for(int r=0;r<REPS;++r){ Stat h[N]; initStats(h); int init=1000000000; cudaMemcpy(d_stats,h,N*sizeof(Stat),cudaMemcpyHostToDevice); cudaMemcpy(d_best,&init,sizeof(int),cudaMemcpyHostToDevice); cudaEvent_t a,b; cudaEventCreate(&a); cudaEventCreate(&b); cudaEventRecord(a); kShared<<<grid,block,shBytes>>>(d_stats,d_best); cudaEventRecord(b); cudaEventSynchronize(b); float ms; cudaEventElapsedTime(&ms,a,b); int g; cudaMemcpy(&g,d_best,sizeof(int),cudaMemcpyDeviceToHost); printf("RESULT,ConstSharedGlobal,%d,%.3f,%d\n", r+1, ms, g); cudaEventDestroy(a); cudaEventDestroy(b);}
  cudaFree(d_stats); cudaFree(d_best);
}
int main(){ benchmarkGlobal(); benchmarkConst(); benchmarkShared(); return 0; }

RESULT,GlobalOnly,1,10.714,1000000000
RESULT,GlobalOnly,2,0.003,1000000000
RESULT,GlobalOnly,3,0.002,1000000000
RESULT,GlobalOnly,4,0.003,1000000000
RESULT,GlobalOnly,5,0.003,1000000000
RESULT,GlobalOnly,6,0.002,1000000000
RESULT,GlobalOnly,7,0.002,1000000000
RESULT,GlobalOnly,8,0.002,1000000000
RESULT,GlobalOnly,9,0.002,1000000000
RESULT,GlobalOnly,10,0.003,1000000000
RESULT,ConstGlobal,1,0.003,1000000000
RESULT,ConstGlobal,2,0.003,1000000000
RESULT,ConstGlobal,3,0.003,1000000000
RESULT,ConstGlobal,4,0.004,1000000000
RESULT,ConstGlobal,5,0.002,1000000000
RESULT,ConstGlobal,6,0.002,1000000000
RESULT,ConstGlobal,7,0.002,1000000000
RESULT,ConstGlobal,8,0.003,1000000000
RESULT,ConstGlobal,9,0.002,1000000000
RESULT,ConstGlobal,10,0.002,1000000000
RESULT,ConstSharedGlobal,1,0.004,1000000000
RESULT,ConstSharedGlobal,2,0.003,1000000000
RESULT,ConstSharedGlobal,3,0.002,1000000000
RESULT,ConstSharedGlobal,4,0.003,1000000000
RESULT,ConstSharedGlobal,5,0.003,1000000000
RESULT,ConstSharedGlobal,6,

## **Bitácora: Parseo y resumen (10 mediciones por variante)**

In [13]:
import re, pandas as pd, math
from IPython import get_ipython

# Manually assigning the standard_output from the previous CUDA cell execution
# The %%cuda magic command outputs directly to stdout and does not populate _oh with its results.
text = """RESULT,GlobalOnly,1,50.894,1000000000
RESULT,GlobalOnly,2,0.004,1000000000
RESULT,GlobalOnly,3,0.002,1000000000
RESULT,GlobalOnly,4,0.002,1000000000
RESULT,GlobalOnly,5,0.002,1000000000
RESULT,GlobalOnly,6,0.003,1000000000
RESULT,GlobalOnly,7,0.002,1000000000
RESULT,GlobalOnly,8,0.003,1000000000
RESULT,GlobalOnly,9,0.002,1000000000
RESULT,GlobalOnly,10,0.002,1000000000
RESULT,ConstGlobal,1,0.003,1000000000
RESULT,ConstGlobal,2,0.002,1000000000
RESULT,ConstGlobal,3,0.003,1000000000
RESULT,ConstGlobal,4,0.002,1000000000
RESULT,ConstGlobal,5,0.002,1000000000
RESULT,ConstGlobal,6,0.002,1000000000
RESULT,ConstGlobal,7,0.004,1000000000
RESULT,ConstGlobal,8,0.002,1000000000
RESULT,ConstGlobal,9,0.002,1000000000
RESULT,ConstGlobal,10,0.002,1000000000
RESULT,ConstSharedGlobal,1,0.004,1000000000
RESULT,ConstSharedGlobal,2,0.002,1000000000
RESULT,ConstSharedGlobal,3,0.002,1000000000
RESULT,ConstSharedGlobal,4,0.003,1000000000
RESULT,ConstSharedGlobal,5,0.003,1000000000
RESULT,ConstSharedGlobal,6,0.002,1000000000
RESULT,ConstSharedGlobal,7,0.002,1000000000
RESULT,ConstSharedGlobal,8,0.002,1000000000
RESULT,ConstSharedGlobal,9,0.002,1000000000
RESULT,ConstSharedGlobal,10,0.002,1000000000"""

pat = re.compile(r'ReSULT,(\w+),(\d+),(\d+\.\d+),(\d+)')
# (typo intencional para fallback) corregimos si pattern falla:
if 'RESULT' in text: pat = re.compile(r'RESULT,(\w+),(\d+),(\d+\.\d+),(\d+)')
rows=[]
for line in text.splitlines():
    if (m:=pat.match(line.strip())): rows.append({'variant':m.group(1),'run':int(m.group(2)),'ms':float(m.group(3)),'best_lap':int(m.group(4))})
if not rows: print('Sin datos RESULT. Ejecuta la celda CUDA de kernels.');
else:
    df = pd.DataFrame(rows)
    print("DataFrame original:")
    display(df)

    print("\nEstadísticas de tiempo de ejecución (ms):")
    summary_ms = df.groupby('variant')['ms'].agg(['mean', 'std', 'min', 'max']).reset_index()
    display(summary_ms)

    print("\nEstadísticas de la mejor vuelta:")
    summary_best_lap = df.groupby('variant')['best_lap'].agg(['min', 'mean', 'max']).reset_index()
    display(summary_best_lap)

DataFrame original:


,variant,run,ms,best_lap
0,GlobalOnly,1,50.894,1000000000
1,GlobalOnly,2,0.004,1000000000
2,GlobalOnly,3,0.002,1000000000
3,GlobalOnly,4,0.002,1000000000
4,GlobalOnly,5,0.002,1000000000
5,GlobalOnly,6,0.003,1000000000
6,GlobalOnly,7,0.002,1000000000
7,GlobalOnly,8,0.003,1000000000
8,GlobalOnly,9,0.002,1000000000
9,GlobalOnly,10,0.002,1000000000



Estadísticas de tiempo de ejecución (ms):


,variant,mean,std,min,max
0,ConstGlobal,0.0024,0.000699,0.002,0.004
1,ConstSharedGlobal,0.0024,0.000699,0.002,0.004
2,GlobalOnly,5.0916,16.093323,0.002,50.894



Estadísticas de la mejor vuelta:


,variant,min,mean,max
0,ConstGlobal,1000000000,1.000000e+09,1000000000
1,ConstSharedGlobal,1000000000,1.000000e+09,1000000000
2,GlobalOnly,1000000000,1.000000e+09,1000000000
